In [9]:
import json
import pandas as pd
import numpy as np
from glob import glob
import os   
from collections import defaultdict
from pprint import pprint
import torch
from huggingface_hub import login
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split

In [10]:
def get_device():
    """Detect best available device on Mac"""
    if torch.cuda.is_available():
        device = "cuda"
        print("✅ Using CUDA GPU")
    elif torch.backends.mps.is_available():
        device = "mps"
        print("✅ Using Apple Silicon MPS")
    else:
        device = "cpu"
        print("⚠️  Using CPU (this will be slow)")
    
    return device




In [ ]:
TAXONOMY = {
    'A': {
        'adaptive': [
            ' (1) Calm/ laid back',
            ' (3) Sad, Emotional pain grieving',
            ' (5) Content, happy, joy, hopeful',
            ' (7) Vigor / energetic',
            ' (9) Justifiable anger/ assertive',
            ' anger, justifiable outrage',
            '(11) Proud',
            '(13) Feel loved, belong'
            ],
        'maladaptive': [
            '(2) Anxious/ fearful/ tense',
            '(4) Depressed, despair, hopeless',
            '(6) Mania',
            '(8) Apathic, don’t care, blunted',
            '(10) Angry (aggression), disgust contempt'
            '(12) Ashamed, guilty',
            '(14) Feel lonely'
            ]
          },
    'B-S': {
        'adaptive': [
            '(1) Self care and improvement'
            ],
        'maladaptive': [
            '(2) Self harm, neglect and avoidance'
            ]
    },
    'B-O': {
        'adaptive': 
            [
            '(1) Relating behavior',
            '(3) Autonomous or adaptive control behavior'
            ],
        'maladaptive': [
            '(2) Fight or flight behavior',
            '(4) Over controlled or controlling behavior'
            ]
    },
    'C-S': 
        {
        'adaptive': 
            [
            '(1) Self-acceptance and compassion'
            ],
          'maladaptive': 
              [
              '(2) Self criticism'
              ]
        },
    'C-O': 
        {
            'adaptive': 
            [
                '(1) Perception of the other as related',
                '(3) Perception of the other as facilitating autonomy needs'
            ],
              'maladaptive': 
              [
                  '(4) Perception of the other as blocking autonomy needs',
                  '(2) Perception of the other as detached or over attached'
                  ]
          },
    'D': 
    {
        'adaptive': 
            [
            '(5) Competence, self esteem, self-care',
            '(1) Relatedness',
            '(3) Autonomy and adaptive control'
            ],
      'maladaptive': 
          [
              '(6) Expectation that competence needs will not be met',
              '(4) Expectation that autonomy needs will not be met',
              '(2) Expectation that relatedness needs will not be met'
          ]
    }
}


def get_taxonomy_string():
    """Format taxonomy for prompt"""
    lines = []
    for dim, categories in TAXONOMY.items():
        lines.append(f"\n**{dim}**")
        lines.append("Adaptive:")
        for cat in categories['adaptive']:
            lines.append(f"  - {cat}")
        lines.append("Maladaptive:")
        for cat in categories['maladaptive']:
            lines.append(f"  - {cat}")
    return "\n".join(lines)

In [12]:
class CLPsychDataLoader:
    """Load CLPsych data with proper ordering"""
    
    def __init__(self, input_dir, split='train'):

        self.split = split
        if split == 'train':
            self.input_dir = os.path.join(input_dir, 'train')
        elif split == 'val':
            self.input_dir = os.path.join(input_dir, 'valid')
        elif split == 'test':
            self.input_dir = os.path.join(input_dir, 'test')
        else:
            raise ValueError("Split must be one of 'train', 'val', or 'test'")
        self.df = None

    def load_clpsych_data(self, filepath):
        with open(filepath, 'r') as f:
            data = json.load(f)
            id, post = data['timeline_id'], data['posts']
        return id, post

    def load(self):
        """Load and parse JSON data into sorted DataFrame"""

        train_posts = []
        for file in glob(os.path.join(self.input_dir, '*.json')):
            # print(f"Loading {file}...")
            id, posts = self.load_clpsych_data(file)
            print(f"Loaded {id} with {len(posts)} posts.")
            for post in posts:
                # print(post)
                try:
                    assert 'post_id' in post
                    assert 'post' in post
                    assert 'evidence' in post
                except AssertionError:
                    print(f"Timeline {id}, Post {post['post_id']} is missing required fields.")
                    continue
                train_posts.append({
                    'timeline_id': id,
                    'post_id': post['post_id'],
                    'post_index': post['post_index'],
                    'text': post['post'],
                    'well_being': post.get('Well-being', 0),
                    'is_switch': post.get('Switch', 0),
                    'is_escalation': post.get('Escalation', 0),
                    'evidence': post['evidence']
                })
        
        # Create DataFrame and sort by timeline_id and post_index
        self.df = pd.DataFrame(train_posts)
        self.df = self.df.sort_values(['timeline_id', 'post_index'])
        
        print(f"Loaded {len(self.df)} posts from {self.df['timeline_id'].nunique()} timelines")
        
        return self.df
    
    def verify_order(self):
        """Verify posts are in correct order within each timeline"""
        print("\n=== Verifying Post Order ===")
        issues = []
        
        for timeline_id in self.df['timeline_id'].unique():
            timeline_posts = self.df[self.df['timeline_id'] == timeline_id]
            indices = timeline_posts['post_index'].tolist()
            
            # Check if indices are in ascending order
            if indices != sorted(indices):
                issues.append(f"Timeline {timeline_id}: {indices}")
        
        if issues:
            print("❌ Order issues found:")
            for issue in issues:
                print(f"  {issue}")
            return False
        else:
            print("✅ All posts are in correct order")
            return True
    
    def get_stats(self):
        """Print dataset statistics"""
        print("\n=== Dataset Statistics ===")
        print(f"Total timelines: {self.df['timeline_id'].nunique()}")
        print(f"Total posts: {len(self.df)}")
        print(f"Avg posts per timeline: {len(self.df) / self.df['timeline_id'].nunique():.2f}")
        
        # print(f"\nSwitch events: {self.df['is_switch'].sum()} ({self.df['is_switch'].mean()*100:.1f}%)")
        # print(f"Escalation events: {self.df['is_escalation'].sum()} ({self.df['is_escalation'].mean()*100:.1f}%)")
        
        # ABCD presence
        print("\n=== ABCD Element Presence ===")
        adaptive_counts = defaultdict(int)
        maladaptive_counts = defaultdict(int)
        
        for _, row in self.df.iterrows():
            evidence = row['evidence']
            
            # Adaptive
            if 'adaptive-state' in evidence:
                for dim in ['A', 'B-S', 'B-O', 'C-S', 'C-O', 'D']:
                    if dim in evidence['adaptive-state'] and evidence['adaptive-state'][dim].get('Category'):
                        adaptive_counts[dim] += 1
            
            # Maladaptive
            if 'maladaptive-state' in evidence:
                for dim in ['A', 'B-S', 'B-O', 'C-S', 'C-O', 'D']:
                    if dim in evidence['maladaptive-state'] and evidence['maladaptive-state'][dim].get('Category'):
                        maladaptive_counts[dim] += 1
        
        print("\nAdaptive:")
        for dim in ['A', 'B-S', 'B-O', 'C-S', 'C-O', 'D']:
            print(f"  {dim}: {adaptive_counts[dim]}")
        
        print("\nMaladaptive:")
        for dim in ['A', 'B-S', 'B-O', 'C-S', 'C-O', 'D']:
            print(f"  {dim}: {maladaptive_counts[dim]}")

# Load data
# train_loader = CLPsychDataLoader('tasks12/', split='train')
# val_loader = CLPsychDataLoader('tasks12/', split='val')
# # test_loader = CLPsychDataLoader('tasks12/', split='test')
# df = val_loader.load()
# val_loader.verify_order()
# val_loader.get_stats()

In [ ]:
def format_evidence_as_json(evidence):
    """Convert evidence dict to clean JSON string"""
    output = {
        'adaptive-state': {},
        'maladaptive-state': {}
    }
    
    # Process adaptive state
    if 'adaptive-state' in evidence:
        for dim, data in evidence['adaptive-state'].items():
            if dim != 'Presence' and isinstance(data, dict):
                if 'Category' in data and data['Category']:
                    output['adaptive-state'][dim] = {
                        'Category': data['Category'],
                        'highlighted_evidence': data.get('highlighted_evidence', '')
                    }
    
    # Process maladaptive state
    if 'maladaptive-state' in evidence:
        for dim, data in evidence['maladaptive-state'].items():
            if dim != 'Presence' and isinstance(data, dict):
                if 'Category' in data and data['Category']:
                    output['maladaptive-state'][dim] = {
                        'Category': data['Category'],
                        'highlighted_evidence': data.get('highlighted_evidence', '')
                    }
    
    return json.dumps(output, indent=2)

def create_instruction_dataset(df):
    """
    Convert DataFrame to instruction-tuning format
    Maintains order from sorted DataFrame
    """
    
    instruction = """You are an expert in mental health text analysis using the MIND framework. Analyze the social media post and identify ABCD (Affect, Behavior, Cognition, Desire) elements.

        For each post, identify:
        - Adaptive self-state elements (supporting well-being)
        - Maladaptive self-state elements (negative patterns)

        ABCD Dimensions:
        - A: Affect (Emotional tone or mood)
        - B-S: Behavior toward Self (Actions or tendencies directed inward)
        - B-O: Behavior toward Others (Actions or tendencies directed outward)
        - C-S: Cognition toward Self (C-S) (Beliefs, interpretations, and appraisals about self)
        - C-O: Cognition toward Others (C-O) (Beliefs, interpretations, and appraisals about others)
        - D: Desire (Motivations, needs, wishes, and expectations)

        Categories:
        """ + get_taxonomy_string() + """

        Output JSON format:
        {
        "adaptive-state": {
            "dimension": {
            "Category": "(number) category name",
            "highlighted_evidence": "exact quote from post"
            }
        },
        "maladaptive-state": {
            "dimension": {
            "Category": "(number) category name",
            "highlighted_evidence": "exact quote from post"
            }
        }
        }

        Rules:
        1. Each dimension can appear in adaptive, maladaptive, both, or neither
        2. Evidence must be exact quote from the post (3-15 words)
        3. Only include dimensions you detect in the post
        4. Multiple dimensions can be present simultaneously"""

    dataset = []
    
    # Iterate through sorted DataFrame
    for idx, row in df.iterrows():
        # Skip posts without evidence
        if not row.get('evidence'):
            continue
        
        # Check if there's actual content
        has_content = False
        evidence = row['evidence']
        
        for state in ['adaptive-state', 'maladaptive-state']:
            if state in evidence:
                for dim, data in evidence[state].items():
                    if dim != 'Presence' and isinstance(data, dict):
                        if data.get('Category'):
                            has_content = True
                            break
        
        if not has_content:
            continue
        
        dataset.append({
            'timeline_id': row['timeline_id'],
            'post_index': row['post_index'],
            'post_id': row['post_id'],
            'instruction': instruction,
            'input': f"Post: {row['text']}",
            'output': format_evidence_as_json(row['evidence'])
        })
    
    # Convert back to DataFrame to maintain order
    dataset_df = pd.DataFrame(dataset)
    dataset_df = dataset_df.sort_values(['timeline_id', 'post_index']).reset_index(drop=True)
    
    print(f"\nCreated {len(dataset_df)} instruction examples")
    print(f"From {dataset_df['timeline_id'].nunique()} timelines")
    
    return dataset_df

# Create instruction dataset
# instruction_df = create_instruction_dataset(df)

# Verify first few examples
# print("\n=== First 3 Examples ===")
# for idx in range(min(3, len(instruction_df))):
#     row = instruction_df.iloc[idx]
#     print(f"\nExample {idx+1}:")
#     print(f"Timeline: {row['timeline_id']}, Post: {row['post_index']}")
#     print(f"Input: {row['input'][:100]}...")
#     print(f"Output: {row['output'][:200]}...")

In [6]:
def timeline_aware_split(instruction_df, test_size=0.15, random_state=42):
    """
    Split by timeline while preserving post order
    """
    # Get unique timeline IDs
    timeline_ids = instruction_df['timeline_id'].unique().tolist()
    
    # Split timeline IDs
    train_timeline_ids, val_timeline_ids = train_test_split(
        timeline_ids,
        test_size=test_size,
        random_state=random_state
    )
    
    # Filter by timeline
    train_df = instruction_df[instruction_df['timeline_id'].isin(train_timeline_ids)].copy()
    val_df = instruction_df[instruction_df['timeline_id'].isin(val_timeline_ids)].copy()
    
    # Ensure order is maintained
    train_df = train_df.sort_values(['timeline_id', 'post_index']).reset_index(drop=True)
    val_df = val_df.sort_values(['timeline_id', 'post_index']).reset_index(drop=True)
    
    print(f"\n=== Train/Val Split ===")
    print(f"Train: {len(train_df)} posts from {len(train_timeline_ids)} timelines")
    print(f"Val: {len(val_df)} posts from {len(val_timeline_ids)} timelines")
    
    # Verify order in both sets
    print("\nVerifying order in train set...")
    for timeline_id in train_timeline_ids[:2]:
        timeline_posts = train_df[train_df['timeline_id'] == timeline_id]
        indices = timeline_posts['post_index'].tolist()
        print(f"  Timeline {timeline_id}: {indices}")
        assert indices == sorted(indices), f"Order broken in train for {timeline_id}!"
    
    print("\nVerifying order in val set...")
    for timeline_id in val_timeline_ids[:2]:
        timeline_posts = val_df[val_df['timeline_id'] == timeline_id]
        indices = timeline_posts['post_index'].tolist()
        print(f"  Timeline {timeline_id}: {indices}")
        assert indices == sorted(indices), f"Order broken in val for {timeline_id}!"
    
    print("✅ Order verified in both train and val sets")
    
    return train_df, val_df

# Split data
# train_df, val_df = timeline_aware_split(instruction_df, test_size=0.15)

In [14]:
def df_to_training_format(df):
    """
    Convert DataFrame to list of dicts for training
    Order is already preserved in DataFrame
    """
    training_data = []
    
    for idx, row in df.iterrows():
        training_data.append({
            'instruction': row['instruction'],
            'input': row['input'],
            'output': row['output']
        })
    
    return training_data

# Convert to training format
# train_data = df_to_training_format(train_df)
# val_data = df_to_training_format(val_df)

# print(f"\nTrain data: {len(train_data)} examples")
# print(f"Val data: {len(val_data)} examples")

# # Save for inspection
# print("\n=== Saving sample data ===")
# with open('train_sample.json', 'w') as f:
#     json.dump(train_data[:5], f, indent=2)
print("Saved first 5 training examples to train_sample.json")

Saved first 5 training examples to train_sample.json


In [15]:
from torch.utils.data import Dataset
import torch

class ABCDInstructionDataset(Dataset):
    """
    Dataset for ABCD instruction tuning
    Preserves order from input list
    """
    
    def __init__(self, data, tokenizer, max_length=2048):
        """
        Args:
            data: List of dicts with 'instruction', 'input', 'output'
                  (already in correct order)
            tokenizer: Hugging Face tokenizer
            max_length: Maximum sequence length
        """
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        """Get item at index (maintains order)"""
        item = self.data[idx]
        
        # Format as Llama-3 chat format
        messages = [
            {"role": "system", "content": item['instruction']},
            {"role": "user", "content": item['input']},
            {"role": "assistant", "content": item['output']}
        ]
        
        # Apply chat template
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        
        # Tokenize
        encodings = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        
        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze(),
            'labels': encodings['input_ids'].squeeze()
        }

# Note: We'll create the actual dataset after loading the model
# to avoid loading tokenizer twice

In [ ]:
from unsloth import FastLanguageModel

device = get_device()

# login(token=hf_token)

# ========== STEP 1: Load Data ==========
print("=" * 60)
print("STEP 1: Loading Data")
print("=" * 60)

train_loader = CLPsychDataLoader('tasks12/', split='train')
val_loader = CLPsychDataLoader('tasks12/', split='val')
train_df = train_loader.load()
print("Training Set Stats")
val_df = val_loader.load()
train_loader.verify_order()
train_loader.get_stats()
print("\n" + "=" * 60)
print("Validation Set Stats")
val_loader.verify_order()
val_loader.get_stats()
# ========== STEP 2: Create Instruction Dataset ==========
print("\n" + "=" * 60)
print("STEP 2: Creating Instruction Dataset")
print("=" * 60)

train_df = create_instruction_dataset(train_df)
val_df = create_instruction_dataset(val_df)

# ========== STEP 3: Train/Val Split ==========
print("\n" + "=" * 60)
print("STEP 3: Train/Val Split")
print("=" * 60)

# train_df, val_df = timeline_aware_split(instruction_df, test_size=0.15)

# Convert to list format
train_data = df_to_training_format(train_df)
val_data = df_to_training_format(val_df)

# ========== STEP 4: Load Model ==========
print("\n" + "=" * 60)
print("STEP 4: Loading Llama-3-8B Model")
print("=" * 60)

model_name = "mlx-community/Meta-Llama-3-8B-8bit"

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="mlx-community/Meta-Llama-3-8B-8bit",
    max_seq_length=1024,
    dtype=torch.float16,
    load_in_4bit=True,
)

# Load tokenizer
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model WITHOUT BitsAndBytesConfig
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B", load_in_4bit=True)
model = model.to(device)
# model.config.use_cache = False
model.config.pretraining_tp = 1

# ========== STEP 5: Setup LoRA ==========
print("\n" + "=" * 60)
print("STEP 5: Configuring LoRA")
print("=" * 60)

model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=8,  # Rank
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ========== STEP 6: Create Datasets ==========
print("\n" + "=" * 60)
print("STEP 6: Creating PyTorch Datasets")
print("=" * 60)

train_dataset = ABCDInstructionDataset(train_data, tokenizer, max_length=1024)
val_dataset = ABCDInstructionDataset(val_data, tokenizer, max_length=1024)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset: {len(val_dataset)} examples")

# ========== STEP 7: Training Configuration ==========
print("\n" + "=" * 60)
print("STEP 7: Configuring Training")
print("=" * 60)

# Add this right before training
print("\n" + "="*60)
print("FINAL DEVICE CHECK")
print("="*60)
print(f"Model device: {next(model.parameters()).device}")
print(f"Expected: mps")

# Quick speed test
# import time
# device = next(model.parameters()).device

# x = torch.randn(1000, 1000).to(device)
# start = time.time()
# for _ in range(100):
#     y = x @ x
# if device.type == "mps":
#     torch.mps.synchronize()
# elapsed = time.time() - start
# print(f"GPU matrix multiply (100 iterations): {elapsed:.2f}s")
# print("If > 5s, something is wrong")
# print("="*60 + "\n")

training_args = TrainingArguments(
    output_dir="./llama3-abcd-lora",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,  # 3% of total steps
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    train_sampling_strategy="sequential",  # Ensure sequential sampling to preserve order
    report_to="none",
    save_total_limit=2,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

# ========== STEP 8: Train ==========
print("\n" + "=" * 60)
print("STEP 8: Starting Training")
print("=" * 60)

trainer.train()

# ========== STEP 9: Save Model ==========
print("\n" + "=" * 60)
print("STEP 9: Saving Model")
print("=" * 60)

trainer.save_model("./llama3-abcd-lora-final")
tokenizer.save_pretrained("./llama3-abcd-lora-final")

print("\n✅ Training complete! Model saved to ./llama3-abcd-lora-final")

/tmp/ipykernel_1094722/1784401024.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Using CUDA GPU
STEP 1: Loading Data
Loaded 83997cd4e7 with 5 posts.
Loaded 8e0a58cfbd with 19 posts.
Loaded cf9aadf596 with 11 posts.
Loaded 9f2e135af8 with 25 posts.
Loaded 5da839acb5 with 10 posts.
Loaded 43f395b896 with 11 posts.
Loaded b1d762ee9f with 9 posts.
Loaded a35b41d1ca with 7 posts.
Loaded 7d9d2e0e0a with 6 posts.
Loaded 306d938d4b with 11 posts.
Loaded 8fc3f6c07d with 9 posts.
Loaded feb7900b62 with 24 posts.
Loaded 91b6a42835 with 13 posts.
Loaded 62defd8df2 with 8 posts.
Loaded 3db8573df5 with 9 posts.
Loaded 6c9677b482 with 11 posts.
Loaded 8c5231d642 with 19 posts.
Loaded d0fb4b962e with 10 posts.
Loaded 30c9e21337 with 11 posts.
Loaded 15564c5c62 with 11 posts.
Loaded d237ea4269 with 12 posts.
Loaded 8080ac11fc with 17 posts.
Loaded aa310342f5 with 20 posts.
Loaded 87821f81b9 with 5 posts.
Loaded caee9a5e58 with 10 posts.
Loaded 

Loading weights: 100%|██████████| 291/291 [00:01<00:00, 157.89it/s, Materializing param=model.norm.weight]                              
Unsloth: Will load unsloth/llama-3-8b-instruct-bnb-4bit as a legacy tokenizer.



STEP 5: Configuring LoRA


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 191 | Num Epochs = 10 | Total steps = 120
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 20,971,520 of 8,051,232,768 (0.26% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,1.382744,0.386659
2,0.475249,0.365095
3,0.455561,0.363850
4,0.397039,0.393966
5,0.219513,0.448204
6,0.173829,0.499250
7,0.104667,0.581231
8,0.075611,0.593656
9,0.044566,0.645436
10,0.024498,0.650528


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
--- Logging error ---
Traceback (most recent call last):
  File "/home/snath/miniconda3/envs/lus/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/home/snath/miniconda3/envs/lus/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/home/snath/miniconda3/envs/lus/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/home/snath/miniconda3/envs/lus/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call s


STEP 9: Saving Model

✅ Training complete! Model saved to ./llama3-abcd-lora-final


In [17]:
#!/usr/bin/env python3
"""
Generate predictions on validation dataset and calculate metrics
"""

import json
import warnings
warnings.filterwarnings('ignore')

import torch
from unsloth import FastLanguageModel
from tqdm import tqdm
import re
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, classification_report

# ========== 1. Load Trained Model ==========
print("="*60)
print("Loading trained model...")
print("="*60)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="llama3-abcd-lora-final",  # Your checkpoint directory
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)
print("✅ Model loaded\n")

# ========== 2. Load Validation Data ==========
print("="*60)
print("Loading validation data...")
print("="*60)

# Load your val_data (should have instruction, input, output format)
# with open('val_data.json', 'r') as f:
#     val_data = json.load(f)

print(f"✅ Loaded {len(val_data)} validation examples\n")

# ========== 3. Generate Predictions ==========
print("="*60)
print("Generating predictions...")
print("="*60)

def predict_single(instruction, input_text, model, tokenizer):
    """Generate prediction for single example"""
    
    # Format as chat
    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": input_text}
    ]
    
    # Apply chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    
    # Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        top_p=0.9,
        use_cache=True,
    )
    
    # Decode
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    
    return response

def parse_json_output(text):
    """Extract JSON from model output"""
    json_match = re.search(r'\{.*\}', text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except:
            return None
    return None

# Generate predictions
predictions = []

for item in tqdm(val_data, desc="Predicting"):
    try:
        # Generate
        response = predict_single(
            item['instruction'],
            item['input'],
            model,
            tokenizer
        )
        
        # Parse
        prediction = parse_json_output(response)
        ground_truth = parse_json_output(item['output'])
        
        predictions.append({
            'prediction': prediction,
            'ground_truth': ground_truth,
            'raw_response': response
        })
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        predictions.append({
            'prediction': None,
            'ground_truth': None,
            'error': str(e)
        })

print(f"\n✅ Generated {len(predictions)} predictions\n")

# Save predictions
with open('val_predictions.json', 'w') as f:
    json.dump(predictions, f, indent=2)
print("✅ Predictions saved to val_predictions.json\n")

# ========== 4. Convert to Binary Matrices ==========
print("="*60)
print("Converting to binary format...")
print("="*60)

# Define all dimensions
dimensions = ['A', 'B-S', 'B-O', 'C-S', 'C-O', 'D']
categories = []
for dim in dimensions:
    categories.append(f"{dim}_adaptive")
    categories.append(f"{dim}_maladaptive")

n_categories = len(categories)

def evidence_to_binary(evidence_dict):
    """Convert evidence dict to binary vector"""
    vector = np.zeros(n_categories)
    
    if not evidence_dict:
        return vector
    
    # Adaptive state
    if 'adaptive-state' in evidence_dict:
        for dim, data in evidence_dict['adaptive-state'].items():
            if dim in dimensions and data:
                idx = categories.index(f"{dim}_adaptive")
                vector[idx] = 1
    
    # Maladaptive state
    if 'maladaptive-state' in evidence_dict:
        for dim, data in evidence_dict['maladaptive-state'].items():
            if dim in dimensions and data:
                idx = categories.index(f"{dim}_maladaptive")
                vector[idx] = 1
    
    return vector

# Convert all predictions and ground truths
y_true = []
y_pred = []

valid_count = 0
for pred in predictions:
    if pred['prediction'] and pred['ground_truth']:
        y_true.append(evidence_to_binary(pred['ground_truth']))
        y_pred.append(evidence_to_binary(pred['prediction']))
        valid_count += 1

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(f"Valid predictions: {valid_count}/{len(predictions)}")
print(f"Binary matrix shape: {y_true.shape}\n")

# ========== 5. Calculate Metrics ==========
print("="*60)
print("EVALUATION RESULTS")
print("="*60)

# Overall metrics (micro-averaged)
precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
    y_true, y_pred, average='micro', zero_division=0
)

print("\n--- Overall (Micro-Averaged) ---")
print(f"Precision: {precision_micro:.4f}")
print(f"Recall:    {recall_micro:.4f}")
print(f"F1 Score:  {f1_micro:.4f}")

# Macro-averaged (average across all dimensions)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, average='macro', zero_division=0
)

print("\n--- Macro-Averaged (All Dimensions) ---")
print(f"Precision: {precision_macro:.4f}")
print(f"Recall:    {recall_macro:.4f}")
print(f"F1 Score:  {f1_macro:.4f}")

# Per-dimension metrics
print("\n--- Per-Dimension Metrics ---")
precision_per_cat, recall_per_cat, f1_per_cat, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0
)

for i, cat in enumerate(categories):
    if support[i] > 0:  # Only show categories that exist in ground truth
        print(f"\n{cat}:")
        print(f"  Precision: {precision_per_cat[i]:.4f}")
        print(f"  Recall:    {recall_per_cat[i]:.4f}")
        print(f"  F1:        {f1_per_cat[i]:.4f}")
        print(f"  Support:   {int(support[i])}")

# Exact match accuracy
from sklearn.metrics import accuracy_score
exact_match = accuracy_score(y_true, y_pred)
print(f"\n--- Exact Match Accuracy ---")
print(f"Accuracy: {exact_match:.4f}")

print("\n" + "="*60)

# ========== 6. Save Results ==========
results = {
    'micro': {
        'precision': float(precision_micro),
        'recall': float(recall_micro),
        'f1': float(f1_micro)
    },
    'macro': {
        'precision': float(precision_macro),
        'recall': float(recall_macro),
        'f1': float(f1_macro)
    },
    'per_dimension': {
        categories[i]: {
            'precision': float(precision_per_cat[i]),
            'recall': float(recall_per_cat[i]),
            'f1': float(f1_per_cat[i]),
            'support': int(support[i])
        }
        for i in range(len(categories))
    },
    'exact_match': float(exact_match)
}

with open('evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n✅ Results saved to evaluation_results.json")

Loading trained model...
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.558 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:01<00:00, 178.32it/s, Materializing param=model.norm.weight]                              
Unsloth: Will load unsloth/llama-3-8b-instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.4 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ Model loaded

Loading validation data...
✅ Loaded 45 validation examples

Generating predictions...


Predicting: 100%|██████████| 45/45 [07:16<00:00,  9.69s/it]


✅ Generated 45 predictions

✅ Predictions saved to val_predictions.json

Converting to binary format...
Valid predictions: 43/45
Binary matrix shape: (43, 12)

EVALUATION RESULTS

--- Overall (Micro-Averaged) ---
Precision: 0.5690
Recall:    0.5562
F1 Score:  0.5625

--- Macro-Averaged (All Dimensions) ---
Precision: 0.4936
Recall:    0.4860
F1 Score:  0.4632

--- Per-Dimension Metrics ---

A_adaptive:
  Precision: 0.5000
  Recall:    0.2000
  F1:        0.2857
  Support:   5

A_maladaptive:
  Precision: 0.8621
  Recall:    0.8065
  F1:        0.8333
  Support:   31

B-S_adaptive:
  Precision: 0.4706
  Recall:    0.4706
  F1:        0.4706
  Support:   17

B-S_maladaptive:
  Precision: 0.5000
  Recall:    0.6250
  F1:        0.5556
  Support:   8

B-O_adaptive:
  Precision: 0.5000
  Recall:    0.5556
  F1:        0.5263
  Support:   18

B-O_maladaptive:
  Precision: 0.0000
  Recall:    0.0000
  F1:        0.0000
  Support:   3

C-S_adaptive:
  Precision: 0.3571
  Recall:    0.7143
  F